# Chain

###  Build a chain with a chat-
                    - messages
                    - models
                    - tools
    - execute tool call in graph nodes.

<img src="../Images/chain_image.png" width="600" height="600">
                    

## Messages
- Messages are fundamental unit of context . It determines how LLms understand context.
- represent the input and output of models
- carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Messages are objects that contain:
 Role - Identifies the message type (e.g. system, user)
 Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
 Metadata - Optional fields such as response information, message IDs, and token usage
LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

## Each message can be supplied with a few things:

    content - content of the message
    name - optionally, a message author
    response_metadata - optionally, a dict of metadata (e.g., often populated by model provider for AIMessages)

In [23]:
import sys
#sys.executable

In [1]:
from langchain_core.messages import AIMessage , HumanMessage, SystemMessage 

messages =[SystemMessage("you are a oacean mammals reasearch expert")]
messages.append(AIMessage(content=f"So you said you were researching ocean mammals?", name="Model"))
messages.append(HumanMessage(content=f"Yes, that's right.",name="Lance"))
messages.append(AIMessage(content=f"Great, what would you like to learn about.", name="Model"))
messages.append(HumanMessage(content=f"I want to learn about the best place to see Orcas in the US.", name="Lance"))

for message in  messages:
    #print(message.content)
    message.pretty_print()

================================ System Message ================================

you are a oacean mammals reasearch expert
================================== Ai Message ==================================
Name: Model

So you said you were researching ocean mammals?
================================ Human Message =================================
Name: Lance

Yes, that's right.
================================== Ai Message ==================================
Name: Model

Great, what would you like to learn about.
================================ Human Message =================================
Name: Lance

I want to learn about the best place to see Orcas in the US.


## Chat Models
Use LLM models as chat models
messages act as input and system instruction 
- Connect API to use genai model
- use gemini-2.5-flash for free tier
- LangChain uses a different internal API version.
To make it work with Gemini 1.5 models, you MUST set:os.environ["GOOGLE_API_USE_V1"] = "true"


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GOOGLE_API_USE_V1"] = "true"

In [3]:
# Configure google‑genai (the new SDK)

from google import genai

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])

for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2

In [12]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

result = llm.invoke(messages)
print(result.content)

Excellent question! As an expert in ocean mammals, I can tell you that finding the "best" place often depends on what kind of experience you're looking for and what time of year you plan to go. However, there are definitely a few standout locations in the US for Orca sightings.

Here are the top contenders:

1.  **Pacific Northwest (Washington State - San Juan Islands & Puget Sound)**
    *   **Why it's often considered the best:** This region is world-renowned for its Orca populations. It's home to the **Southern Resident Killer Whales (SRKW)**, an endangered population that primarily feeds on salmon. While their presence has become less predictable due to salmon scarcity, this area also sees a significant and increasing number of **Transient (Bigg's) Killer Whales**, which hunt marine mammals like seals, sea lions, and porpoises.
    *   **Best Time:**
        *   **Summer (June - September):** Traditionally the best time for SRKW, though their movements are now less reliable.
      

# Obtain Metadata

In [16]:
result

AIMessage(content='Excellent question! As an expert in ocean mammals, I can tell you that finding the "best" place often depends on what kind of experience you\'re looking for and what time of year you plan to go. However, there are definitely a few standout locations in the US for Orca sightings.\n\nHere are the top contenders:\n\n1.  **Pacific Northwest (Washington State - San Juan Islands & Puget Sound)**\n    *   **Why it\'s often considered the best:** This region is world-renowned for its Orca populations. It\'s home to the **Southern Resident Killer Whales (SRKW)**, an endangered population that primarily feeds on salmon. While their presence has become less predictable due to salmon scarcity, this area also sees a significant and increasing number of **Transient (Bigg\'s) Killer Whales**, which hunt marine mammals like seals, sea lions, and porpoises.\n    *   **Best Time:**\n        *   **Summer (June - September):** Traditionally the best time for SRKW, though their movements

In [14]:
result.response_metadata

{'finish_reason': 'STOP',
 'model_name': 'gemini-2.5-flash',
 'safety_ratings': [],
 'model_provider': 'google_genai'}

In [15]:
result.usage_metadata

{'input_tokens': 57,
 'output_tokens': 2353,
 'total_tokens': 2410,
 'input_token_details': {'cache_read': 0},
 'output_token_details': {'reasoning': 1217}}

# Tool Binding and Calling
- Tools are useful whenever you want a model to interact with external systems.
- When tool is binded to an llm, model get aware of input schema or payload of that perticular tool
- based on the user input, modle decided to call the tool and provides the api i/p as tool schema
    i.e, llm o/p to the tool -> tool in/p schema or payload



## Task:  call a function to multiply two numbers

In [20]:
# createa tool function with description
def multiply(a: int,b: int)->int:
    """ Multiply provided two numbers i,e, a and b 
    args: 
    a = first a int
    b = second int 
    result is 'c' which is an int
    """    
    c= a*b
    return c

# bind the tool function
llm_with_tool = llm.bind_tools([multiply])

tool_call = llm_with_tool.invoke([HumanMessage(role ="user" , content="What is the result of multiplying 23 and 45" , name = "diya")])    
    

In [21]:
tool_call

AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"b": 45, "a": 23}'}, '__gemini_function_call_thought_signatures__': {'56f1b427-8f56-416a-8186-b8017b798b8d': 'Cs4BAb4+9vsDvp/xp/aJSZEzA+zpGCNtP5/q2qowVHGXvVoR4ToBTkMSbxFaNu+g7NoRnK1jeXpbpgOlfOgmyfTJbqH/FMFnbK799IYMl5GlKQ8JXgG6XCjR9Qb7wX1dP0XrOTwM3VZrwqR8LjvTqQOnt1UNpgOi8DkRBsZGw/BEBD4ZLn824u1AYpX5shF31XiY9H/zqHDqpXntDpnYCrMiigQ2Cxxw/qFbWD+i+TNFQ6JD9j/GzKYIqGgPpnMiadqrr3e30hLlw1c+MlgzKSU='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d846b-91ba-78f0-9b01-2edfebfce037-0', tool_calls=[{'name': 'multiply', 'args': {'b': 45, 'a': 23}, 'id': '56f1b427-8f56-416a-8186-b8017b798b8d', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 97, 'output_tokens': 71, 'total_tokens': 168, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 51}})

In [22]:
tool_call.tool_calls

[{'name': 'multiply',
  'args': {'b': 45, 'a': 23},
  'id': '56f1b427-8f56-416a-8186-b8017b798b8d',
  'type': 'tool_call'}]